# ASL from Hand Landmarks

In [1]:
# Load data

import pandas as pd

landmarks_df = pd.read_csv("../data/preprocessed/handlandmarks_d01megadataset.csv")

In [2]:
landmarks_df.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,57,58,59,60,61,62,letter_index,dataset,user,image_path
0,0,0.0,0.0,0.0,0.542589,-0.123035,-0.099435,0.928112,-0.550039,-0.125472,...,-0.263594,-0.677533,-0.352767,-0.247789,-0.623774,-0.302888,0,A03_kaggle_asl,NaN,../data/dataset/D01_megadataset/A/A03_kaggle_a...
1,1,0.0,0.0,0.0,0.531273,-0.145492,-0.093864,0.904783,-0.576911,-0.117419,...,-0.298376,-0.715572,-0.312834,-0.269324,-0.616285,-0.254807,0,A03_kaggle_asl,NaN,../data/dataset/D01_megadataset/A/A03_kaggle_a...
2,2,0.0,0.0,0.0,0.549435,-0.172507,-0.104700,0.933805,-0.542889,-0.128209,...,-0.286453,-0.950963,-0.240619,-0.256799,-0.785396,-0.175428,0,A03_kaggle_asl,NaN,../data/dataset/D01_megadataset/A/A03_kaggle_a...
3,3,0.0,0.0,0.0,0.058546,-0.293606,0.038537,0.351927,-0.601428,0.112847,...,0.900312,-1.290697,-0.077889,0.786906,-1.463245,-0.067724,0,A03_kaggle_asl,NaN,../data/dataset/D01_megadataset/A/A03_kaggle_a...
4,4,0.0,0.0,0.0,0.058546,-0.293606,0.038537,0.351927,-0.601428,0.112847,...,0.900312,-1.290697,-0.077889,0.786906,-1.463245,-0.067724,0,A03_kaggle_asl,NaN,../data/dataset/D01_megadataset/A/A03_kaggle_a...


In [7]:
landmarks_df["letter_index"].nunique()

24

In [3]:
# Count images per dataset
landmarks_df["dataset"].value_counts()

dataset
A03_kaggle_asl           107719
A04_mandalay_aslhg        24000
B02_aslyset                4800
B01_original_aslmnist      1680
Name: count, dtype: int64

In [ ]:
landmarks_df

In [ ]:
# No hand detected


# Stratify by dataset and letter


In [ ]:
train_paths, train_labels = find_images(TRAIN_ROOT)

sample_indices = np.random.default_rng(42).choice(
    len(train_paths),
    size=min(6, len(train_paths)),
    replace=False,
)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax in axes.flat:
    ax.axis("off")

with create_landmarker() as landmarker:
    for ax, index in zip(axes.flat, sample_indices):
        rgb = read_rgb(train_paths[index])
        _, landmarks = get_features(rgb, landmarker)

        ax.imshow(rgb)

        if landmarks:
            height, width = rgb.shape[:2]
            ax.scatter(
                [point.x * width for point in landmarks],
                [point.y * height for point in landmarks],
                s=12,
                c="lime",
            )
            status = "detected"
        else:
            status = "NO HAND"

        ax.set_title(f"{LETTERS[train_labels[index]]}: {status}")

plt.tight_layout()
plt.show()

In [ ]:
train_indices, val_indices = train_test_split(
    np.arange(len(train_paths)),
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

# Extract once, then select the corresponding rows.
features_all, training_failures = extract_features(train_paths)

X_train = features_all[train_indices]
y_train = train_labels[train_indices]

X_val = features_all[val_indices]
y_val = train_labels[val_indices]

train_ok = np.isfinite(X_train).all(axis=1)

print(f"Training hand detection: {train_ok.mean():.1%}")
print("First loading/detection failures:", training_failures[:3])

# All 24 classes must have usable training examples.
missing = set(range(24)) - set(y_train[train_ok])

if missing:
    raise ValueError(
        f"No usable training images for: {[LETTERS[i] for i in sorted(missing)]}"
    )

In [ ]:
classifier = XGBClassifier(
    objective="multi:softprob",
    num_class=24,
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=4,
)

classifier.fit(
    X_train[train_ok],
    y_train[train_ok],
)